# Paradoja del Cumpleaños

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def prob_cump (n):
  prob = 1                  #probabilidad de que n personas no cumplan años el mismo dia
  for i in range(n):
    prob = prob*(1-i/365)
  return (1-prob)*100       #probabilidad de que n personas cumplan años el mismo dia

num_per = np.arange(2,101,1)
prob = np.zeros_like(num_per)
for i in range(len(num_per)):
  prob[i] = prob_cump(num_per[i])

#Gráficamos número de personas - n vs probabilidad - p(n)
plt.figure(figsize=(12,6))
plt.plot(num_per,prob)
plt.title(f'Paradoja del Cumpleaños')
plt.xlabel(f'número de personas - n')
plt.ylabel(f'probabilidad - $p(n)$')
plt.grid()

#Añadimos más puntos en la gráfica
plt.xticks(np.arange(0, 101, 3))
plt.yticks(np.arange(0, 101, 10))

plt.show()

In [ ]:
print(f'La probabilidad de que al menos dos personas cumplan años el mismo dia en un grupo de 23 personas es p(23)={prob_cump(23)}')

In [ ]:
print(f'La probabilidad de que al menos dos personas cumplan años el mismo dia en un grupo de 41 personas es p(41)={prob_cump(41)}')

# Generar Secuencias de ADN Aleatorias

In [ ]:
# !pip install biopython

In [ ]:
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import random as rand

# Por defecto las funciones aceptan y retornan tipo Seq a menos que se especifique tipo = "str"

def generar_adn(longitud, tipo=""):
    sec_adn = list()
    bases = ['A','C','G','T']
    for i in range(longitud):
        sec_adn.append(rand.choice(bases))     # Selecciona aleatoriamente una base
    if tipo == "str":
        return ''.join(sec_adn)
    return Seq(''.join(sec_adn))

In [ ]:
print(f'La secuencia de ADN generada es: {generar_adn(50)}')

# Generar Secuencias de ADN Mutadas

In [ ]:
import random as rand

def mutar_adn(secuencia, tasa_mutacion, tipo=""):      #0 <= tasa_mutacion <= 1
    bases = ['A', 'C', 'G', 'T']
    secuencia_mutada = list(secuencia)
    for i in range(len(secuencia_mutada)):
        if rand.random() < tasa_mutacion:     #Condición de para mutar una base de la secuencia
            # Seleccionamos las bases distintas para hacer la mutación
            opciones = [b for b in bases if b != secuencia_mutada[i]]
            secuencia_mutada[i] = rand.choice(opciones)
    if tipo == "str":
        return ''.join(secuencia_mutada)
    return Seq(''.join(secuencia_mutada))

In [ ]:
g = generar_adn(50)
print(f'La secuencia de ADN generada es: {g}')
g_mutada = mutar_adn(g,0.1)
print(f'La secuencia de ADN mutada a un {0.1*100}% es: {g_mutada}')

# Construcción del Conjunto de los k-meros

In [ ]:
def k_meros(secuencia, k, tipo=""): # 1 <= k <= len(secuencia)
  kmeros= set()
  if k > len(secuencia):
    return None
  elif k == len(secuencia):
    kmeros.add(str(secuencia)) if tipo == "str" else kmeros.add(secuencia)
    return kmeros
  else:
    for i in range(len(secuencia) - (k - 1)):
      kmeros.add(str(secuencia[i:i+k])) if tipo == "str" else kmeros.add(secuencia[i:i+k])
    return kmeros

In [ ]:
g = generar_adn(30)
print(f'La secuencia de ADN generada es g = {g}')
#construimos el conjunto de los k-meros de g
k=15
kmeros = k_meros(g,k)
print(f'El conjunto de {k}-meros de g es:')
for k in kmeros:
  print(k)

# Construcción de la Matriz de Representación

In [ ]:
import numpy as np

def matrizRepresentacion (conjuntos):
  elementos = set()
  for s in conjuntos:
    elementos.update(s)

  M = np.zeros((len(elementos),len(conjuntos)))
  row = 0
  for i in elementos:
    column = 0
    for s in conjuntos:
      if i in s:
        M[row,column] = 1
      column += 1
    row += 1
  return [elementos,M]

In [ ]:
# Generamos secuencias de ADN aleatorias

adn_1 = generar_adn(20)
adn_2 = generar_adn(20)
adn_3 = generar_adn(20)
adn_4 = generar_adn(20)

j=1
for i in [adn_1,adn_2,adn_3,adn_4]:
  print(f'Decuencia de ADN generada adn_{j} = {i}')
  j += 1

In [ ]:
# Construimos el conjunto de k-meros (k=3) de las secuencias de ADN

k = 3
S1 = k_meros(adn_1,k)
S2 = k_meros(adn_2,k)
S3 = k_meros(adn_3,k)
S4 = k_meros(adn_4,k)

In [ ]:
# Construimos la matriz de representación

elementos, M = matrizRepresentacion([S1,S2,S3,S4])
for i in elementos:
  print(i)

In [ ]:
elementos = set()
elementos.update(S1,S2,S3,S4)
for i in elementos:
  print(i)

In [ ]:
a = set(['a','b','c'])
b = set([9,8,7])
a.update(b)
print(a)
print(len(a))
print('a' in b)

In [ ]:
print(np.zeros((4,2)))

# Aplicación a datos reales

In [ ]:
import pandas as pd

# Importamos la información obtenida del NCBI que detalla
# las características de cada secuenciación completa del
# SARS-CoV-2 como la especie, variante, origen de la muestra, etc.

df = pd.read_csv("sequences.csv")
print(df.dtypes)

In [ ]:
# Le cambiamos el tipo a la fecha de recolección y de liberación
df["Collection_Date"] = pd.to_datetime(df["Collection_Date"], format="mixed")
df["Release_Date"] = pd.to_datetime(df["Release_Date"], format="mixed")

In [ ]:
# Las secuenciaciones corresponden a diferentes variantes (Pangolin)
# de las cuales existen múltiples registros

print(df["Pangolin"].value_counts())

In [ ]:
# Las secuencias propiamente las encontramos en el
# archivo sequences.fasta

for record in SeqIO.parse("sequences.fasta", "fasta"):
  print(record.id)
  print(repr(record.seq))
  print(len(record.seq))

# Es de resaltar que la longitud de las secuencias es de alrededor de 30000 bases cada una

In [ ]:
# donde record.id corresponde al atributo Accession de
# la tabla, el cual podemos utilizar como clave primaria
# cuando decidamos contrastar las mutaciones entre variantes

df["Accession"].is_unique

In [ ]:
# Filtramos por la primera aparicion de cada nueva variante

df.sort_values("Collection_Date").groupby("Pangolin").head(1)

In [ ]:
# Nos quedamos con unas pocas para el análisis

g_id = df.sort_values("Collection_Date").groupby("Pangolin").head(1)["Accession"].head(5).array
print(g_id)
g_id = set(g_id)

g = [sec_record.seq for sec_record in SeqIO.parse("sequences.fasta", "fasta") if sec_record.id.split(".")[0] in g_id]
for s in g:
  print(repr(s))

In [ ]:
# Obtenemos los k-meros

k = 13
S = [k_meros(seq, k) for seq in g]
print(S)

In [ ]:
# Calculamos la matriz de representación

elementos, M = matrizRepresentacion(S)
for i in elementos:
  print(i)